In [0]:
%pip install -U pyyaml numpy torch scikit-learn mlflow joblib
dbutils.library.restartPython()

In [0]:
import yaml
import os
import numpy as np

config_path = "config.yaml" if os.path.exists("config.yaml") else "../config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

CATALOG = config["storage"]["catalog_name"]
SCHEMA = config["storage"]["schema_name"]
VOLUME = config["storage"]["volume_name"]
HF_DATASET = config["dataset"]["hf_name"]
EMBEDDING_DIM = config["models"]["embedding_dim"]
NUM_CLASSES = 10

# Katalog w Volume na wagi modeli między kolejnymi paczkami
CHECKPOINT_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/model_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 1. Sprawdzamy numer bieżącej paczki
state_row = spark.sql(f"""
    SELECT last_batch_id FROM {CATALOG}.{SCHEMA}.pipeline_state 
    WHERE dataset_name = '{HF_DATASET}' ORDER BY last_batch_id DESC LIMIT 1
""").collect()[0]
current_batch_id = state_row["last_batch_id"] + 1

# 2. Pobieramy wektory z bieżącej paczki
train_rows = spark.table(f"{CATALOG}.{SCHEMA}.silver_current_batch").collect()

# Jeśli to paczka #2 lub kolejna, dokładamy próbki z Replay Buffera (przeciw zapominaniu!)
if current_batch_id > 1 and spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.silver_replay_buffer"):
    replay_rows = spark.table(f"{CATALOG}.{SCHEMA}.silver_replay_buffer").collect()
    train_rows = train_rows + replay_rows
    print(f"Dołączono {len(replay_rows)} wektorów z Replay Buffera.")

X_train = np.array([r["features"] for r in train_rows], dtype=np.float32)
y_train = np.array([r["label_id"] for r in train_rows], dtype=np.int64)

# 3. Pobieramy stały, zbalansowany zbiór testowy (500 zdjęć)
test_rows = spark.table(f"{CATALOG}.{SCHEMA}.silver_test_holdout").orderBy("test_idx").collect()
X_test = np.array([r["features"] for r in test_rows], dtype=np.float32)
y_test = np.array([r["label_id"] for r in test_rows], dtype=np.int64)

# Mapa ID klasy -> nazwa klasy (np. 0 -> AnnualCrop)
label_to_name = {r["label_id"]: r["class_name"] for r in test_rows}

print(f"Paczka #{current_batch_id} gotowa do nauki!")
print(f"Rozmiar zbioru uczącego: {X_train.shape}, Rozmiar stałego zbioru testowego: {X_test.shape}")

In [0]:
import time
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Model 1: Prosta sieć liniowa (1 warstwa: 576 -> 10)
class LinearHead(nn.Module):
    def __init__(self, input_dim=576, num_classes=10):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)
        
    def forward(self, x):
        return self.fc(x)

# Model 2: Głębsza sieć wielowarstwowa MLP (576 -> 256 -> 10)
class DeepMLPHead(nn.Module):
    def __init__(self, input_dim=576, hidden_dim=256, dropout=0.3, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
        
    def forward(self, x):
        return self.net(x)

def run_pytorch_model(model, model_name, epochs=12, base_lr=0.002):
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"{model_name}.pt")
    loaded_from_prev = False
    
    # Jeśli istnieją wagi z poprzedniej paczki -> wczytujemy je i douczamy
    if current_batch_id > 1 and os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path))
        loaded_from_prev = True
        lr = base_lr * 0.5
    else:
        lr = base_lr
        
    dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    loader = DataLoader(dataset, batch_size=64, shuffle=True)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # 1. Pomiar czasu treningu + zbieranie średniego błędu treningowego (train_loss)
    t0_train = time.time()
    model.train()
    epoch_losses = []
    for epoch in range(epochs):
        batch_losses = []
        for bx, by in loader:
            optimizer.zero_grad()
            out = model(bx)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())
        epoch_losses.append(float(np.mean(batch_losses)))
        
    train_time = round(time.time() - t0_train, 4)
    final_train_loss = round(epoch_losses[-1], 4)
    
    # Zapisujemy wagi na dysk dla kolejnej paczki
    torch.save(model.state_dict(), ckpt_path)
    
    # 2. Pomiar czasu inferencji (testowania) na 500 zdjęciach
    t0_eval = time.time()
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_test))
        probs = torch.softmax(logits, dim=1).numpy()
    eval_time = round(time.time() - t0_eval, 4)
    
    return probs, final_train_loss, train_time, eval_time, loaded_from_prev, ckpt_path, lr

print("Funkcje treningowe i pomiarowe gotowe!")

In [0]:
import joblib
import mlflow
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, 
    log_loss, precision_recall_fscore_support
)
from datetime import datetime

user_email = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{user_email}/EuroSAT_Models_Comparison")

# Konfiguracja wszystkich porównywanych modeli i ich hiperparametrów
models_config = [
    {
        "model_name": "PyTorch_Linear_Head",
        "family": "PyTorch",
        "epochs": 12,
        "base_lr": 0.002,
        "arch_desc": "Linear(576->10)"
    },
    {
        "model_name": "PyTorch_Deep_MLP",
        "family": "PyTorch",
        "epochs": 15,
        "base_lr": 0.0015,
        "arch_desc": "MLP(576->256->10, BN, Dropout=0.3)"
    },
    {
        "model_name": "Sklearn_SGD_LogReg",
        "family": "Scikit-Learn",
        "epochs": 5,
        "base_lr": 0.01,
        "arch_desc": "SGDClassifier(loss=log_loss, partial_fit)"
    }
]

runs_records = []          # Tabela 1: Ogólne metryki, czasy, błędy i parametry każdego modelu
class_metrics_records = [] # Tabela 2: Błędy i skuteczność każdego modelu w rozbiciu na 10 klas
predictions_records = []   # Tabela 3: Wszystkie pojedyncze predykcje każdego modelu

for cfg in models_config:
    m_name = cfg["model_name"]
    
    with mlflow.start_run(run_name=f"Batch_{current_batch_id}_{m_name}") as run:
        if m_name == "PyTorch_Linear_Head":
            net = LinearHead(EMBEDDING_DIM, NUM_CLASSES)
            probs, tr_loss, tr_time, ev_time, is_inc, ckpt_path, used_lr = run_pytorch_model(
                net, m_name, epochs=cfg["epochs"], base_lr=cfg["base_lr"]
            )
        elif m_name == "PyTorch_Deep_MLP":
            net = DeepMLPHead(EMBEDDING_DIM, 256, 0.3, NUM_CLASSES)
            probs, tr_loss, tr_time, ev_time, is_inc, ckpt_path, used_lr = run_pytorch_model(
                net, m_name, epochs=cfg["epochs"], base_lr=cfg["base_lr"]
            )
        elif m_name == "Sklearn_SGD_LogReg":
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"{m_name}.pkl")
            if current_batch_id > 1 and os.path.exists(ckpt_path):
                clf = joblib.load(ckpt_path)
                is_inc = True
            else:
                clf = SGDClassifier(loss="log_loss", learning_rate="adaptive", eta0=cfg["base_lr"], random_state=42)
                is_inc = False
                
            t0_tr = time.time()
            for _ in range(cfg["epochs"]):
                clf.partial_fit(X_train, y_train, classes=np.arange(NUM_CLASSES))
            tr_time = round(time.time() - t0_tr, 4)
            
            # Średni błąd Log-Loss na zbiorze treningowym
            train_probs = np.clip(clf.predict_proba(X_train), 1e-7, 1 - 1e-7)
            tr_loss = round(float(log_loss(y_train, train_probs, labels=np.arange(NUM_CLASSES))), 4)
            joblib.dump(clf, ckpt_path)
            
            t0_ev = time.time()
            probs = clf.predict_proba(X_test)
            ev_time = round(time.time() - t0_ev, 4)
            used_lr = cfg["base_lr"]

        # Zabezpieczenie prawdopodobieństw przed log(0)
        probs_clipped = np.clip(probs, 1e-7, 1 - 1e-7)
        preds = np.argmax(probs, axis=1)
        confs = np.max(probs, axis=1)
        
        # 1. Wyliczamy ogólne metryki i średnie błędy dla modelu
        acc = round(float(accuracy_score(y_test, preds)), 4)
        err_rate = round(1.0 - acc, 4)  # Średni błąd klasyfikacji (odsetek pomyłek)
        tst_loss = round(float(log_loss(y_test, probs_clipped, labels=np.arange(NUM_CLASSES))), 4)
        f1_mac = round(float(f1_score(y_test, preds, average="macro")), 4)
        prec_mac = round(float(precision_score(y_test, preds, average="macro", zero_division=0)), 4)
        rec_mac = round(float(recall_score(y_test, preds, average="macro", zero_division=0)), 4)
        
        correct_mask = (preds == y_test)
        avg_conf_correct = round(float(confs[correct_mask].mean()), 4) if np.any(correct_mask) else 0.0
        avg_conf_error = round(float(confs[~correct_mask].mean()), 4) if np.any(~correct_mask) else 0.0
        
        # Logowanie do MLflow
        mlflow.log_params({
            "batch_id": current_batch_id,
            "model_name": m_name,
            "family": cfg["family"],
            "architecture": cfg["arch_desc"],
            "epochs": cfg["epochs"],
            "learning_rate": used_lr,
            "incremental_loaded": is_inc,
            "train_samples": len(X_train)
        })
        mlflow.log_metrics({
            "train_loss": tr_loss,
            "test_log_loss": tst_loss,
            "test_error_rate": err_rate,
            "test_accuracy": acc,
            "test_f1_macro": f1_mac,
            "test_precision_macro": prec_mac,
            "test_recall_macro": rec_mac,
            "train_time_sec": tr_time,
            "inference_time_sec": ev_time
        })
        mlflow.log_artifact(ckpt_path, artifact_path="checkpoints")
        
        now_ts = datetime.utcnow()
        
        # Zapis do Tabeli 1 (Ogólne przebiegi modeli)
        runs_records.append({
            "batch_id": int(current_batch_id),
            "model_name": m_name,
            "model_family": cfg["family"],
            "architecture": cfg["arch_desc"],
            "epochs": int(cfg["epochs"]),
            "learning_rate": float(used_lr),
            "incremental_loaded": bool(is_inc),
            "train_samples": int(len(X_train)),
            "train_time_sec": float(tr_time),
            "inference_time_sec": float(ev_time),
            "total_time_sec": round(float(tr_time + ev_time), 4),
            "train_loss": float(tr_loss),
            "test_log_loss": float(tst_loss),
            "test_error_rate": float(err_rate),
            "test_accuracy": float(acc),
            "test_f1_macro": float(f1_mac),
            "test_precision_macro": float(prec_mac),
            "test_recall_macro": float(rec_mac),
            "avg_conf_when_correct": float(avg_conf_correct),
            "avg_conf_when_wrong": float(avg_conf_error),
            "mlflow_run_id": run.info.run_id,
            "evaluated_at": now_ts
        })
        
        # 2. Wyliczamy metryki i średnie błędy per klasa dla tego modelu (Tabela 2)
        p_cls, r_cls, f1_cls, _ = precision_recall_fscore_support(
            y_test, preds, labels=np.arange(NUM_CLASSES), zero_division=0
        )
        for c_id in range(NUM_CLASSES):
            c_mask = (y_test == c_id)
            c_acc = float(np.mean(preds[c_mask] == y_test[c_mask]))
            c_err = round(1.0 - c_acc, 4)
            # Średni błąd Log-Loss dla zdjęć tej konkretnej klasy
            c_loss = float(np.mean(-np.log(probs_clipped[c_mask, c_id])))
            
            class_metrics_records.append({
                "batch_id": int(current_batch_id),
                "model_name": m_name,
                "label_id": int(c_id),
                "class_name": label_to_name.get(c_id, f"class_{c_id}"),
                "class_accuracy": round(c_acc, 4),
                "class_error_rate": c_err,
                "class_log_loss": round(c_loss, 4),
                "class_precision": round(float(p_cls[c_id]), 4),
                "class_recall": round(float(r_cls[c_id]), 4),
                "class_f1": round(float(f1_cls[c_id]), 4),
                "evaluated_at": now_ts
            })
            
        # 3. Zbieramy wszystkie pojedyncze predykcje dla tego modelu (Tabela 3)
        for idx, r in enumerate(test_rows):
            true_id = int(r["label_id"])
            pred_id = int(preds[idx])
            sample_loss = float(-np.log(probs_clipped[idx, true_id]))
            
            predictions_records.append({
                "batch_id": int(current_batch_id),
                "model_name": m_name,
                "test_idx": int(r["test_idx"]),
                "filename": r["filename"],
                "true_label_id": true_id,
                "true_class_name": r["class_name"],
                "pred_label_id": pred_id,
                "pred_class_name": label_to_name.get(pred_id, f"class_{pred_id}"),
                "is_correct": bool(true_id == pred_id),
                "classification_error": int(0 if true_id == pred_id else 1),
                "sample_log_loss": round(sample_loss, 4),
                "confidence": round(float(confs[idx]), 4),
                "evaluated_at": now_ts
            })
            
        print(f"[{m_name}] Acc: {acc*100:.2f}% | Błąd (Error Rate): {err_rate*100:.2f}% | Test Loss: {tst_loss:.4f} | Czas nauki: {tr_time}s")

In [0]:
# 1. Zapisujemy ogólne metryki, czasy, parametry i średnie błędy (gold_model_runs)
runs_df = spark.createDataFrame(runs_records)
runs_df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.gold_model_runs")

# 2. Zapisujemy błędy i skuteczność w rozbiciu na klasy dla każdego modelu (gold_model_class_metrics)
class_df = spark.createDataFrame(class_metrics_records)
class_df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.gold_model_class_metrics")

# 3. Zapisujemy pojedyncze predykcje wszystkich modeli (gold_model_predictions)
preds_df = spark.createDataFrame(predictions_records)
preds_df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.gold_model_predictions")

print("Zapisano komplet danych dla wszystkich modeli w 3 tabelach warstwy Gold!")

# Podgląd Tabeli 1: Porównanie czasów, błędów i metryk wszystkich modeli
display(
    spark.sql(f"""
        SELECT 
            batch_id,
            model_name,
            incremental_loaded,
            train_loss,
            test_log_loss,
            test_error_rate,
            test_accuracy,
            test_f1_macro,
            train_time_sec,
            inference_time_sec
        FROM {CATALOG}.{SCHEMA}.gold_model_runs
        WHERE batch_id = {current_batch_id}
        ORDER BY model_name
    """)
)